In [1]:
!pkill -f "scspecies-env"

: 

In [1]:

!pkill -f "scspecies-env30/bin/python\|scspecies-env40/bin/python"

In [ ]:
！pkill code

SyntaxError: invalid syntax (582840860.py, line 1)

In [ ]:
echo 'vm.swappiness=10' | sudo tee -a /etc/sysctl.conf

In [ ]:
!conda install -y cuml=24.10 cuda-version=12.8 -c rapidsai -c nvidia -c conda-forge

Retrieving notices: done
Channels:
 - rapidsai
 - nvidia
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.0
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/i/miniconda3/envs/scspecies-env30

  added / updated specs:
    - cuda-version=12.8
    - cuml=24.10


The following NEW packages will be INSTALLED:

  aom                conda-forge/linux-64::aom-3.9.1-hac33072_0 
  attr               conda-forge/linux-64::attr-2.5.2-hb03c661_1 
  aws-c-auth         conda-forge/linux-64::aws-c-auth-0.9.0-h0fbd49f_19 
  aws-c-cal          conda-forge/linux-64::aws-c-cal-0.9.2-he7b75e1_1 
  aws-c-common       conda-forge/linux-64::aws-c-common-0.12.4-hb03c661_0 
  aws-c-compression  conda-forge/linux-64::aws-c-compression-0.3.1-h92c474e_6 
  aws-c-event-stream conda-forge/linux-64::aws-c-ev

In [3]:
pip freeze > env_cuml.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from gprofiler import GProfiler


pip install gprofiler-official

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (
    homogeneity_score, silhouette_score, davies_bouldin_score,
    adjusted_rand_score, calinski_harabasz_score
)
import warnings
warnings.filterwarnings('ignore')

# ===================== 生物学注释 =====================
biological_annotation = {
    'B cell': 'Immune',
    'Kupffer cell': 'Immune',
    'T cell': 'Immune',
    'dendritic cell': 'Immune',
    'granulocyte': 'Immune',
    'macrophage': 'Immune',
    'mature NK T cell': 'Immune',
    'monocyte': 'Immune',
    'natural killer cell': 'Immune',
    'neutrophil': 'Immune',
    'cardiac muscle cell': 'Muscle',
    'endocardial cell': 'Endothelium',
    'endothelial cell': 'Endothelium',
    'hepatic stellate cell': 'Epithelium',
    'hepatocyte': 'Epithelium',
    'kidney tubule cell': 'Epithelium',
    'pericyte': 'Endothelium',
    'exocrine cell': 'Secretory',
    'pancreatic A cell': 'Secretory',
    'pancreatic D cell': 'Secretory',
    'pancreatic PP cell': 'Secretory',
    'type B pancreatic cell': 'Secretory',
    'fibroblast': 'Stroma',
    'fibrocyte': 'Stroma',
    'Schwann cell': 'Stroma',
    'unknown': 'Unknown'
}

# ===================== 快速纯度计算 =====================
def purity(true, pred):
    """
    计算聚类纯度：每个聚类中占比最高的真实类别数 / 总样本数
    """
    tab = pd.crosstab(pd.Series(true), pd.Series(pred))
    return tab.max(axis=0).sum() / tab.sum().sum()

# ===================== 极速评估主函数 =====================
def evaluate_biological_clustering_fast(adata, cell_type_key='cell_type', basis='z_mu'):
    """
    评估单细胞聚类结果的生物学一致性
    
    参数：
    adata: AnnData对象，包含细胞特征和注释
    cell_type_key: str，adata.obs中细胞类型列名
    basis: str，adata.obsm中用于聚类的特征矩阵key
    
    返回：
    dict，核心评估指标
    """
    # 提取特征矩阵和细胞类型
    if basis not in adata.obsm:
        raise ValueError(f"adata.obsm中未找到{basis}，请检查basis参数是否正确")
    X = adata.obsm[basis]
    
    if cell_type_key not in adata.obs:
        raise ValueError(f"adata.obs中未找到{cell_type_key}，请检查cell_type_key参数是否正确")
    cell_types = adata.obs[cell_type_key].values

    # 映射生物学大类（补充未定义的细胞类型为Unknown）
    adata.obs['biological_category'] = adata.obs[cell_type_key].map(
        lambda x: biological_annotation.get(x, 'Unknown')
    )
    bio_cat = adata.obs['biological_category'].values
    bio_cats = np.unique(bio_cat)

    # 基于生物学大类数量进行KMeans聚类
    n_clusters = len(bio_cats)
    kmeans = KMeans(n_clusters=n_clusters, random_state=0, n_init='auto')
    clusters = kmeans.fit_predict(X)

    # 计算全局评估指标
    ari = adjusted_rand_score(bio_cat, clusters)  # 调整兰德指数（0-1，越高越好）
    ch = calinski_harabasz_score(X, clusters)    # 卡林斯基-哈拉巴斯指数（越高越好）
    homo = homogeneity_score(bio_cat, clusters)  # 同质性（0-1，越高越好）
    # 轮廓系数：采样加速，避免大数据量计算过慢
    sil = silhouette_score(X, clusters, sample_size=min(2000, X.shape[0]))
    db = davies_bouldin_score(X, clusters)       # DB指数（越低越好）
    overall_purity = purity(bio_cat, clusters)   # 整体聚类纯度

    # 计算各生物学大类内的细胞类型一致性
    cat_purity = {}
    cat_n = {}
    for c in bio_cats:
        mask = bio_cat == c
        cat_n[c] = mask.sum()
        # 至少2个细胞才计算纯度，否则默认1.0
        if mask.sum() >= 2:
            cat_purity[c] = purity(cell_types[mask], clusters[mask])
        else:
            cat_purity[c] = 1.0
    avg_bio_consist = np.mean(list(cat_purity.values()))

    # 计算各生物学大类的富集度（该类细胞集中在单个聚类的程度）
    cat_enrich = {}
    for c in bio_cats:
        mask = bio_cat == c
        if mask.sum() == 0:
            cat_enrich[c] = 0.0
        else:
            vc = pd.Series(clusters[mask]).value_counts()
            cat_enrich[c] = vc.max() / vc.sum()  # 最高占比聚类的比例
    avg_enrich = np.mean(list(cat_enrich.values()))

    # 修正：计算各聚类的生物学大类纯度（原逻辑错误，现在计算聚类中主要生物学大类的占比）
    cluster_purity = {}
    for clu in np.unique(clusters):
        mask = clusters == clu
        # 真实逻辑：该聚类中占比最高的生物学大类数量 / 聚类总细胞数
        cluster_purity[f'聚类{clu+1}'] = purity(bio_cat[mask], [clu]*mask.sum())

    # 生成评估报告
    lines = [
        "聚类生物学一致性评估报告",
        "跨来源评估开关: 开启",
        "============================================================",
        "",
        "📈 核心聚类生物学一致性指标",
        f"  类内同质性（Homogeneity）     {homo:.4f}  (越高越好)",
        f"  类间分离度（Calinski-Harabasz）{ch:.2f}  (越高越好)",
        f"  拓扑一致性 ARI               {ari:.4f}  (≥0.7为优秀 | 越高越好)",
        f"  平均生物学大类一致性          {avg_bio_consist:.4f}  (越高越好)",
        "",
        "📊 扩展聚类质量指标",
        f"  平均轮廓系数                 {sil:.4f}  (≥0.5为良好 | 越高越好)",
        f"  Davies-Bouldin指数           {db:.4f}  (≤1.0为优秀 | 越低越好)",
        f"  类群富集度                   {avg_enrich:.4f}  (≥0.8为高度富集 | 越高越好)",
        f"  聚类纯度                     {overall_purity:.4f}  (≥0.8为高纯度 | 越高越好)",
        "",
        "🔍 细分指标（各生物学类群）",
        "  1. 大类内一致性（越高越好）:"
    ]

    # 添加各生物学大类的一致性
    for c in sorted(cat_purity.keys()):
        lines.append(f"    {c} (n={cat_n[c]}): {cat_purity[c]:.4f}")

    # 添加各生物学大类的富集度
    lines += ["  2. 类群富集度 (越高越好):"]
    for c in sorted(cat_enrich.keys()):
        lines.append(f"    {c}: {cat_enrich[c]:.4f}")

    # 基础信息和各聚类纯度
    lines += [
        "",
        "📦 聚类基础信息",
        f"  聚类数                       {n_clusters}",
        f"  生物学类群数                 {len(bio_cats)}",
        "  各聚类纯度 (越高越好):"
    ]
    for name in sorted(cluster_purity.keys()):
        lines.append(f"    {name}: {cluster_purity[name]:.4f}")

    # 综合评分（加权计算）
    # 权重：ARI(0.3) + 轮廓系数(0.2) + 整体纯度(0.2) + 大类一致性(0.2) + (1-DB指数)(0.1)
    score = (
        ari * 0.3 + 
        sil * 0.2 + 
        overall_purity * 0.2 + 
        avg_bio_consist * 0.2 + 
        (1 - db) * 0.1
    ) * 100
    # 综合评级
    if score >= 70:
        grade = "✅ 优秀"
        conclusion = "聚类结果与生物学注释高度一致，聚类质量优秀"
    elif score >= 50:
        grade = "🟡 良好"
        conclusion = "聚类结果与生物学注释基本一致，聚类质量良好"
    else:
        grade = "🔴 一般"
        conclusion = "聚类结果与生物学注释一致性一般，建议优化特征或参数"

    # 补充综合评估到报告
    lines += [
        "",
        "🎯 综合评估",
        f"  综合得分                     {score:.2f}  (满分≈100 | 越高越好)",
        f"  综合评级                     {grade}",
        f"  评估结论                     {conclusion}"
    ]

    # 保存并打印报告
    report = '\n'.join(lines)
    with open(os.path.join(SAVE_DIR,"bio_sgf_metric_enhanced.txt"), "w", encoding="utf-8") as f:
        f.write(report)
    print(report)

    # 返回核心结果
    return {
        "ARI（全局一致性）": round(ari, 4),
        "平均生物学大类一致性": round(avg_bio_consist, 4),
        "综合得分": round(score, 2),
        "综合评级": grade,
        "整体聚类纯度": round(overall_purity, 4),
        "轮廓系数": round(sil, 4),
        "DB指数": round(db, 4)
    }

    
# 调用评估函数
result = evaluate_biological_clustering_fast(
    adata=ADATA,
    cell_type_key='cell_type',
    basis='z_mu'
)

# 打印核心结果汇总
print("\n" + "="*60)
print("核心结果汇总")
print("="*60)
for key, value in result.items():
    print(f"{key}: {value}")